In [31]:
# Ignore warning
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import glob, os
import matplotlib.pyplot as plt
import numpy as np
import geopandas
import netCDF4
import h5py
import datetime as dt
import pyproj

import networkx as nx
import torch_geometric
from torch_geometric.utils.convert import to_networkx, from_networkx
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

# check pytorch version
import torch    
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim    

from tqdm import tqdm

from pyproj import Proj, transform
from shapely.geometry import Polygon
import cartopy.crs as ccrs
import torch

import time

from scipy.interpolate import griddata

import cdsapi
import xarray as xr
from urllib.request import urlopen

from urllib.request import urlretrieve

import pickle

import scipy.io as sio

%load_ext autoreload
%autoreload 2

import dgl
from dgl.data import DGLDataset
from dgl import save_graphs, load_graphs
import torch
import os


from functions import *
from DGL_model import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
import dgl
from dgl.data import DGLDataset
import torch
import os

class KarateClubDataset(DGLDataset):
    def __init__(self):
        super().__init__(name='karate_club')

    def process(self):
        nodes_data = pd.read_csv('./members.csv')
        edges_data = pd.read_csv('./interactions.csv')
        node_features = torch.from_numpy(nodes_data['Age'].to_numpy())
        node_labels = torch.from_numpy(nodes_data['Club'].astype('category').cat.codes.to_numpy())
        edge_features = torch.from_numpy(edges_data['Weight'].to_numpy())
        edges_src = torch.from_numpy(edges_data['Src'].to_numpy())
        edges_dst = torch.from_numpy(edges_data['Dst'].to_numpy())

        self.graph = dgl.graph((edges_src, edges_dst), num_nodes=nodes_data.shape[0])
        self.graph.ndata['feat'] = node_features
        self.graph.ndata['label'] = node_labels
        self.graph.edata['weight'] = edge_features

        # If your dataset is a node classification dataset, you will need to assign
        # masks indicating whether a node belongs to training, validation, and test set.
        n_nodes = nodes_data.shape[0]
        n_train = int(n_nodes * 0.6)
        n_val = int(n_nodes * 0.2)
        train_mask = torch.zeros(n_nodes, dtype=torch.bool)
        val_mask = torch.zeros(n_nodes, dtype=torch.bool)
        test_mask = torch.zeros(n_nodes, dtype=torch.bool)
        train_mask[:n_train] = True
        val_mask[n_train:n_train + n_val] = True
        test_mask[n_train + n_val:] = True
        self.graph.ndata['train_mask'] = train_mask
        self.graph.ndata['val_mask'] = val_mask
        self.graph.ndata['test_mask'] = test_mask

    def __getitem__(self, i):
        return self.graph

    def __len__(self):
        return 1

dataset = KarateClubDataset()
graph = dataset[0]

print(graph)

Graph(num_nodes=34, num_edges=156,
      ndata_schemes={'feat': Scheme(shape=(), dtype=torch.int64), 'label': Scheme(shape=(), dtype=torch.int8), 'train_mask': Scheme(shape=(), dtype=torch.bool), 'val_mask': Scheme(shape=(), dtype=torch.bool), 'test_mask': Scheme(shape=(), dtype=torch.bool)}
      edata_schemes={'weight': Scheme(shape=(), dtype=torch.float64)})


In [29]:
graph.ndata['feat']

tensor([44, 37, 37, 40, 30, 32, 36, 47, 35, 37, 35, 46, 46, 48, 41, 49, 46, 38,
        44, 41, 48, 34, 43, 41, 40, 34, 38, 42, 42, 44, 48, 41, 35, 46])

In [12]:
urllib.request.urlretrieve(
    'https://data.dgl.ai/tutorial/dataset/graph_edges.csv', './graph_edges.csv')
urllib.request.urlretrieve(
    'https://data.dgl.ai/tutorial/dataset/graph_properties.csv', './graph_properties.csv')
edges = pd.read_csv('./graph_edges.csv')
properties = pd.read_csv('./graph_properties.csv')

edges.head()

properties.head()

class SyntheticDataset(DGLDataset):
    def __init__(self):
        super().__init__(name='synthetic')
        
    def process(self):
        edges = pd.read_csv('./graph_edges.csv')
        properties = pd.read_csv('./graph_properties.csv')
        self.graphs = []
        self.labels = []
        
        # Create a graph for each graph ID from the edges table.
        # First process the properties table into two dictionaries with graph IDs as keys.
        # The label and number of nodes are values.
        label_dict = {}
        num_nodes_dict = {}
        for _, row in properties.iterrows():
            label_dict[row['graph_id']] = row['label']
            num_nodes_dict[row['graph_id']] = row['num_nodes']
            
        # For the edges, first group the table by graph IDs.
        edges_group = edges.groupby('graph_id')
        
        # For each graph ID...
        for graph_id in edges_group.groups:
            # Find the edges as well as the number of nodes and its label.
            edges_of_id = edges_group.get_group(graph_id)
            src = edges_of_id['src'].to_numpy()
            dst = edges_of_id['dst'].to_numpy()
            num_nodes = num_nodes_dict[graph_id]
            label = label_dict[graph_id]
            
            # Create a graph and add it to the list of graphs and labels.
            g = dgl.graph((src, dst), num_nodes=num_nodes)
            self.graphs.append(g)
            self.labels.append(label)
            
        # Convert the label list to tensor for saving.
        self.labels = torch.LongTensor(self.labels)
        
    def __getitem__(self, i):
        return self.graphs[i], self.labels[i]
    
    def __len__(self):
        return len(self.graphs)

dataset = SyntheticDataset()
graph, label = dataset[0]
print(graph, label)

52426342it [00:14, 3662405.06it/s]                                                                                     

KeyboardInterrupt



In [2]:
files = glob.glob('D:\\ISSM\\version2\\transient_g10000_y20_r*.mat')

train_list = []
val_list = []

for filename in tqdm(files[:1]):
    
    rate = float(filename[-7:-4])
    test = sio.loadmat(filename)
    
    xc = test['S'][0][0][0]
    yc = test['S'][0][0][1]
    elements = test['S'][0][0][2]-1

    smb = test['S'][0][0][3]
    vx = test['S'][0][0][4]
    vy = test['S'][0][0][5]
    vel = test['S'][0][0][6]
    H = test['S'][0][0][7]
    f = test['S'][0][0][8]

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  8.16it/s]


In [3]:
n_year, n_sample = H.shape

In [4]:
n_year

240

In [166]:
files = glob.glob('D:\\ISSM\\version2\\transient_g5000_y20_r*.mat')

train_list = []
val_list = []

for filename in tqdm(files[:1]):
    
    rate = float(filename[-7:-4])
    test = sio.loadmat(filename)
    elements = test['S'][0][0][2]-1

100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.55s/it]


In [46]:
files = glob.glob('D:\\ISSM\\version2\\transient_y20_r*.mat')

train_list = []
val_list = []

for filename in tqdm(files[:1]):
    
    rate = float(filename[-7:-4])
    test = sio.loadmat(filename)
    
    xc = test['S'][0][0][0]
    yc = test['S'][0][0][1]
    elements = test['S'][0][0][2]-1

    smb = test['S'][0][0][3]
    vx = test['S'][0][0][4]
    vy = test['S'][0][0][5]
    vel = test['S'][0][0][6]
    H = test['S'][0][0][7]
    f = test['S'][0][0][8]

    n_year, n_sample = H.shape

    for t in range(0, n_year):
        
        src = []
        dst = []
        weight = []
        inputs = torch.zeros([n_sample, 4])
        outputs = torch.zeros([n_sample, 6])

        for i in range(0, n_sample): 
            inputs[i, :] = torch.tensor([(xc[i, 0]-xc.min())/(xc.max()-xc.min()), (yc[i, 0]-yc.min())/(yc.max()-yc.min()), rate*0.001, t/n_year])
            outputs[i, :] = torch.tensor([smb[t,i], vx[t, i]/5000, vy[t, i]/5000, vel[t,i]/5000, H[t,i]/4000, f[t,i]/3000])

            p1, p2 = np.where(elements == i)

            for p in p1:
                for k in elements[p]:
                    if k != i:
                        dist = ((xc[i]-xc[k])**2+(yc[i]-yc[k])**2)**0.5
                        weight.append(np.exp(-(dist/1000)))
                        src.append(int(i))
                        dst.append(int(k))
        
        src = torch.tensor(src)
        dst = torch.tensor(dst)
        weight = torch.tensor(weight)
        
        g = dgl.graph((src, dst), num_nodes=n_sample)
        g.ndata['feat'] = inputs
        g.ndata['label'] = outputs
        g.edata['weight'] = weight


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:43<00:00, 43.56s/it]


In [156]:
filename = 'D:\\ISSM\\version2\\transient_HO_g10000_y20_r*.mat'

## Dataset for train ===================================
class PIG_train_Dataset(DGLDataset):
    def __init__(self):
        super(PIG_train_Dataset, self).__init__(name='pig');
        
    def process(self):
        self.graphs = []
        filename = 'D:\\ISSM\\version2\\transient_g5000_y20_r*.mat'
        files = glob.glob(filename)

        for filename in tqdm(files[:]):

            rate = float(filename[-7:-4])
            
            if (rate % 200 != 0) and (rate % 200 != 100):
                
                test = sio.loadmat(filename)

                xc = test['S'][0][0][0]
                yc = test['S'][0][0][1]
                elements = test['S'][0][0][2]-1

                smb = test['S'][0][0][3]
                vx = test['S'][0][0][4]
                vy = test['S'][0][0][5]
                vel = test['S'][0][0][6]
                H = test['S'][0][0][7]
                f = test['S'][0][0][8]

                n_year, n_sample = H.shape          

                for t in range(0, n_year):

                    src = []
                    dst = []
                    weight = []
                    inputs = torch.zeros([n_sample, 5])
                    outputs = torch.zeros([n_sample, 5])

                    for i in range(0, n_sample):        
                        inputs[i, :] = torch.tensor([(xc[i, 0]-xc.min())/(xc.max()-xc.min()), (yc[i, 0]-yc.min())/(yc.max()-yc.min()), rate*0.001, t/n_year, smb[t,i]])
                        outputs[i, :] = torch.tensor([vx[t, i]/5000, vy[t, i]/5000, vel[t,i]/5000, H[t,i]/4000, f[t,i]/3000])

                        p1, p2 = np.where(elements == i)
                        connect = []

                        for p in p1:
                            for k in elements[p]:
                                if (k != i) and (k not in connect):
                                    # connect.append(k)
                                    dist = ((xc[i]-xc[k])**2+(yc[i]-yc[k])**2)**0.5
                                    weight.append(np.exp(-(dist/1000)))
                                    src.append(int(i))
                                    dst.append(int(k))

                    src = torch.tensor(src)
                    dst = torch.tensor(dst)
                    weight = torch.tensor(weight)

    #                 train_mask = torch.zeros(n_sample, dtype=torch.bool)
    #                 val_mask = torch.zeros(n_sample, dtype=torch.bool)
    #                 test_mask = torch.zeros(n_sample, dtype=torch.bool)        

    #                 if rate % 100 == 0:
    #                     test_mask[:] = True
    #                 else:
    #                     if t % 10 == 5:
    #                         val_mask[:] = True
    #                     else:
    #                         train_mask[:] = True

                    g = dgl.graph((src, dst), num_nodes=n_sample)
                    g.ndata['feat'] = inputs
                    g.ndata['label'] = outputs
                    g.edata['weight'] = weight

    #                 g.ndata['train_mask'] = train_mask
    #                 g.ndata['val_mask'] = val_mask
    #                 g.ndata['test_mask'] = test_mask

                    self.graphs.append(g)
        
    def __getitem__(self, i):
        return self.graphs[i]
    
    def __len__(self):
        return len(self.graphs)

## Dataset for test ===================================
class PIG_val_Dataset(DGLDataset):
    def __init__(self):
        super(PIG_val_Dataset, self).__init__(name='pig')
        
    def process(self):
        self.graphs = []
        
        filename = 'D:\\ISSM\\version2\\transient_g5000_y20_r*.mat'
        files = glob.glob(filename)

        for filename in tqdm(files[:]):

            rate = float(filename[-7:-4])
            
            if rate % 200 == 100:
            
                test = sio.loadmat(filename)

                xc = test['S'][0][0][0]
                yc = test['S'][0][0][1]
                elements = test['S'][0][0][2]-1

                smb = test['S'][0][0][3]
                vx = test['S'][0][0][4]
                vy = test['S'][0][0][5]
                vel = test['S'][0][0][6]
                H = test['S'][0][0][7]
                f = test['S'][0][0][8]

                n_year, n_sample = H.shape          

                for t in range(0, n_year):

                    src = []
                    dst = []
                    weight = []
                    inputs = torch.zeros([n_sample, 5])
                    outputs = torch.zeros([n_sample, 5])

                    for i in range(0, n_sample):        
                        inputs[i, :] = torch.tensor([(xc[i, 0]-xc.min())/(xc.max()-xc.min()), (yc[i, 0]-yc.min())/(yc.max()-yc.min()), rate*0.001, t/n_year, smb[t,i]])
                        outputs[i, :] = torch.tensor([vx[t, i]/5000, vy[t, i]/5000, vel[t,i]/5000, H[t,i]/4000, f[t,i]/3000])

                        p1, p2 = np.where(elements == i)
                        connect = []

                        for p in p1:
                            for k in elements[p]:
                                if (k != i) and (k not in connect):
                                    # connect.append(k)
                                    dist = ((xc[i]-xc[k])**2+(yc[i]-yc[k])**2)**0.5
                                    weight.append(np.exp(-(dist/1000)))
                                    src.append(int(i))
                                    dst.append(int(k))

                    src = torch.tensor(src)
                    dst = torch.tensor(dst)
                    weight = torch.tensor(weight)

                    g = dgl.graph((src, dst), num_nodes=n_sample)
                    g.ndata['feat'] = inputs
                    g.ndata['label'] = outputs
                    g.edata['weight'] = weight

                    self.graphs.append(g)
        
    def __getitem__(self, i):
        return self.graphs[i]
    
    def __len__(self):
        return len(self.graphs)

## Dataset for test ===================================
class PIG_test_Dataset(DGLDataset):
    def __init__(self):
        super(PIG_test_Dataset, self).__init__(name='pig')
        
    def process(self):
        self.graphs = []
        
        filename = 'D:\\ISSM\\version2\\transient_g5000_y20_r*.mat'
        files = glob.glob(filename)

        for filename in tqdm(files[:]):

            rate = float(filename[-7:-4])
            
            if rate % 200 == 0:
            
                test = sio.loadmat(filename)

                xc = test['S'][0][0][0]
                yc = test['S'][0][0][1]
                elements = test['S'][0][0][2]-1

                smb = test['S'][0][0][3]
                vx = test['S'][0][0][4]
                vy = test['S'][0][0][5]
                vel = test['S'][0][0][6]
                H = test['S'][0][0][7]
                f = test['S'][0][0][8]

                n_year, n_sample = H.shape          

                for t in range(0, n_year):

                    src = []
                    dst = []
                    weight = []
                    inputs = torch.zeros([n_sample, 5])
                    outputs = torch.zeros([n_sample, 5])

                    for i in range(0, n_sample):        
                        inputs[i, :] = torch.tensor([(xc[i, 0]-xc.min())/(xc.max()-xc.min()), (yc[i, 0]-yc.min())/(yc.max()-yc.min()), rate*0.001, t/n_year, smb[t,i]])
                        outputs[i, :] = torch.tensor([vx[t, i]/5000, vy[t, i]/5000, vel[t,i]/5000, H[t,i]/4000, f[t,i]/3000])

                        p1, p2 = np.where(elements == i)
                        connect = []

                        for p in p1:
                            for k in elements[p]:
                                if (k != i) and (k not in connect):
                                    # connect.append(k)
                                    dist = ((xc[i]-xc[k])**2+(yc[i]-yc[k])**2)**0.5
                                    weight.append(np.exp(-(dist/1000)))
                                    src.append(int(i))
                                    dst.append(int(k))

                    src = torch.tensor(src)
                    dst = torch.tensor(dst)
                    weight = torch.tensor(weight)

                    g = dgl.graph((src, dst), num_nodes=n_sample)
                    g.ndata['feat'] = inputs
                    g.ndata['label'] = outputs
                    g.edata['weight'] = weight

                    self.graphs.append(g)
        
    def __getitem__(self, i):
        return self.graphs[i]
    
    def __len__(self):
        return len(self.graphs)
    
train_set = PIG_train_Dataset()
save_graphs("../data/DGL_train_dataset_g5000.bin", train_set.graphs)

val_set = PIG_val_Dataset()
save_graphs("../data/DGL_val_dataset_g5000.bin", val_set.graphs)

test_set = PIG_test_Dataset()
save_graphs("../data/DGL_test_dataset_g5000.bin", test_set.graphs)
print("Done!")

100%|████████████████████████████████████████████████████████████████████████████| 36/36 [2:30:35<00:00, 250.99s/it]


Done!


# GNN data read & Conversion into CNN data (PIG dataset)

## Read ISSM simulation results

In [34]:
filename = 'D:\\ISSM\\version3\\PIG_transient_m10000_r*.mat'
files = glob.glob(filename)

first = True

for filename in tqdm(files[:1]):

    rate = int(filename[-7:-4])*10

    if rate >= 0: #(rate % 200 != 0) and (rate % 200 != 100):

        test = sio.loadmat(filename)

        xc = test['S'][0][0][0]
        yc = test['S'][0][0][1]
        elements = test['S'][0][0][2]-1

        smb = test['S'][0][0][3]
        vx = test['S'][0][0][4]
        vy = test['S'][0][0][5]
        vel = test['S'][0][0][6]
        surface = test['S'][0][0][7]
        base = test['S'][0][0][8]
        H = test['S'][0][0][9]
        f = test['S'][0][0][10]

        n_year, n_sample = H.shape
        
        # [('x'), ('y'), ('elements'), ('smb'), ('Vx'), ('Vy'), ('Vel'), ('surface'), ('base'), ('H'), ('floating'), ('elapsed_time')]

100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.24it/s]


In [7]:
train_files, val_files, test_files = generate_list(region = "PIG", folder = "D:\\ISSM\\version3")

In [9]:
val_files

['D:\\ISSM\\version3\\PIG_transient_m02000_r010.mat',
 'D:\\ISSM\\version3\\PIG_transient_m02000_r030.mat',
 'D:\\ISSM\\version3\\PIG_transient_m02000_r050.mat',
 'D:\\ISSM\\version3\\PIG_transient_m02000_r070.mat',
 'D:\\ISSM\\version3\\PIG_transient_m05000_r010.mat',
 'D:\\ISSM\\version3\\PIG_transient_m05000_r030.mat',
 'D:\\ISSM\\version3\\PIG_transient_m05000_r050.mat',
 'D:\\ISSM\\version3\\PIG_transient_m05000_r070.mat',
 'D:\\ISSM\\version3\\PIG_transient_m10000_r010.mat',
 'D:\\ISSM\\version3\\PIG_transient_m10000_r030.mat',
 'D:\\ISSM\\version3\\PIG_transient_m10000_r050.mat',
 'D:\\ISSM\\version3\\PIG_transient_m10000_r070.mat']

In [20]:
g = GNN_PIG_Dataset(val_files)

  0%|                                                                                   | 0/12 [00:00<?, ?it/s]

D:\ISSM\version3\PIG_transient_m02000_r010.mat 37684


 33%|█████████████████████████                                                  | 4/12 [00:07<00:12,  1.60s/it]

D:\ISSM\version3\PIG_transient_m05000_r010.mat 16700


 67%|██████████████████████████████████████████████████                         | 8/12 [00:11<00:03,  1.05it/s]

D:\ISSM\version3\PIG_transient_m10000_r010.mat 10686


100%|██████████████████████████████████████████████████████████████████████████| 12/12 [00:13<00:00,  1.15s/it]


## Convert GNN data to CNN data

In [83]:
folder = "D:\\ISSM\\version3"
files = glob.glob(f"{folder}\\PIG_transient*.mat")

for filename in files[:]:
    print(filename)
    rate = int(filename.split("_r")[1][:3])
    test = sio.loadmat(filename)

    xc = test['S'][0][0][0]
    yc = test['S'][0][0][1]
    elements = test['S'][0][0][2]-1
    smb = test['S'][0][0][3]
    vx = test['S'][0][0][4]
    vy = test['S'][0][0][5]
    vel = test['S'][0][0][6]
    surface = test['S'][0][0][7]
    base = test['S'][0][0][8]
    H = test['S'][0][0][9]
    f = test['S'][0][0][10]
    # mask = test['S'][0][0][11]
    # ice = np.zeros(mask.shape) # Negative: ice; Positive: no-ice
    # ice[mask > 0] = 0.5 # ice = 0; no-ice = 1
    # ice = np.where(mask < 0, mask / 1000000, mask/10000)

    n_year, n_sample = H.shape        

    coord = np.array([xc[:, 0], yc[:, 0]]).transpose()
    gridx, gridy = np.meshgrid(np.arange(xc.min(), xc.max(), 2000), np.arange(yc.min(), yc.max(), 2000))
    input0 = np.zeros((n_year, 12, gridx.shape[0], gridx.shape[1]))
    output0 = np.zeros((n_year, 6, gridx.shape[0], gridx.shape[1])) 
    mask = griddata(coord, xc[:, 0], (gridx, gridy), method='linear')
    mask[~np.isnan(mask)] = 1
    mask[np.isnan(mask)] = 0

    for t in tqdm(range(0, n_year)):

        # INPUT: x/y coordinates, melting rate, time, SMB, Vx0, Vy0, Surface0, Base0, Thickness0, Floating0
        inputs = torch.zeros([n_sample, 12])
        # OUTPUT: Vx, Vy, Vel, Surface, Thickness, Floating
        outputs = torch.zeros([n_sample, 6])

        ## INPUTS ================================================
        inputs[:, 0] = torch.tensor((xc[:, 0]-xc.min())/10000) # torch.tensor(xc[0, :]/10000) # torch.tensor((xc[:, 0]-xc.min())/(xc.max()-xc.min())) # X coordinate
        inputs[:, 1] = torch.tensor((yc[:, 0]-yc.min())/10000) # torch.tensor(yc[0, :]/10000) # torch.tensor((yc[:, 0]-yc.min())/(yc.max()-yc.min())) # Y coordinate
        inputs[:, 2] = torch.where(torch.tensor(f[0, :]) < 0, rate/100, 0) # Melting rate (0-100)
        inputs[:, 3] = torch.tensor(t/n_year) # Year
        inputs[:, 4] = torch.tensor(smb[t, :]/20) # Surface mass balance
        inputs[:, 5] = torch.tensor(vx[0, :]/10000) # Initial Vx
        inputs[:, 6] = torch.tensor(vy[0, :]/10000) # Initial Vx
        inputs[:, 7] = torch.tensor(vel[0, :]/10000) # Initial Vel
        inputs[:, 8] = torch.tensor(surface[0, :]/5000) # Initial surface elevation
        inputs[:, 9] = torch.tensor(base[0, :]/5000) # Initial base elevation
        inputs[:, 10] = torch.tensor(H[0, :]/5000) # Initial ice thickness
        inputs[:, 11] = torch.tensor(f[0, :]/5000) # Initial floating part
        # inputs[:, 11] = torch.tensor(ice[0, :]) # Initial ice mask

        ## OUTPUTS ===============================================
        outputs[:, 0] = torch.tensor(vx[t, :]/10000) # Initial Vx
        outputs[:, 1] =  torch.tensor(vy[t, :]/10000) # Initial Vx
        outputs[:, 2] = torch.tensor(vel[t, :]/10000) # Initial surface elevation
        outputs[:, 3] = torch.tensor(surface[t, :]/5000) # Initial base elevation
        outputs[:, 4] = torch.tensor(H[t, :]/5000) # Initial ice thickness
        outputs[:, 5] = torch.tensor(f[t, :]/5000) # Initial floating part 
        # outputs[:, 5] = torch.tensor(ice[t, :]) # Initial floating part 

        for c in range(0, inputs.shape[1]):
            input0[t, c, :, :] = griddata(coord, inputs[:, c], (gridx, gridy), method='nearest')
        for c in range(0, outputs.shape[1]):
            output0[t, c, :, :] = griddata(coord, outputs[:, c], (gridx, gridy), method='nearest')
    
    input0 = input0 * mask
    output0 = output0 * mask
    with open(filename.replace(".mat", "_CNN.pkl"), 'wb') as file:
        pickle.dump([input0.astype(np.float16), output0.astype(np.float16)], file)

D:\ISSM\version3\PIG_transient_m02000_r000.mat


100%|████████████████████████████████████████████████████████████████████████| 240/240 [03:55<00:00,  1.02it/s]


D:\ISSM\version3\PIG_transient_m02000_r002.mat


100%|████████████████████████████████████████████████████████████████████████| 240/240 [04:12<00:00,  1.05s/it]


D:\ISSM\version3\PIG_transient_m02000_r004.mat


...

100%|████████████████████████████████████████████████████████████████████████| 240/240 [03:21<00:00,  1.19it/s]


D:\ISSM\version3\PIG_transient_m10000_r068.mat


100%|████████████████████████████████████████████████████████████████████████| 240/240 [03:33<00:00,  1.12it/s]


D:\ISSM\version3\PIG_transient_m10000_r070.mat


100%|████████████████████████████████████████████████████████████████████████| 240/240 [03:22<00:00,  1.19it/s]
